Digital Business University of Applied Sciences

Data Science und Management (M. Sc.)

DAMI01 / DATA01 Data Analytics

Prof. Dr. Daniel Ambach

Julia Schmid (200022)

***
# Time-Series-Clustering
# Teil 1: Business & Data Understanding

***
## Phase 1: Geschäftsverständnis
***

**Problemstellung**

Die Kundensegmentierung ist ein etabliertes Verfahren zur Analyse und Steuerung von Kundenbeziehungen. Klassische Ansätze basieren jedoch überwiegend auf aggregierten Kennzahlen und statischen Merkmalen, wodurch die zeitliche Dynamik des Kundenverhaltens weitgehend unberücksichtigt bleibt. Transaktionsdaten weisen eine inhärente zeitliche Ordnung auf, die sich mit konventionellen Segmentierungsmethoden nur begrenzt nutzen lässt. Es stellt sich daher die Frage, ob sich mithilfe von Time Series Clustering (TSC) auf Basis von Transaktionsdaten Kundengruppen identifizieren lassen, die ähnliche zeitliche Verhaltensmuster aufweisen.

**Ziel der Analyse**

Ziel dieser Analyse ist es, Kunden anhand ihrer zeitlichen Transaktionsverläufe zu segmentieren.
Dazu werden mehrere TSC-Methoden (agglomeratives Clustering, k-Medoids, Spectral Clustering, Fuzzy C-Means und DBSCAN), jeweils mit unterschiedlichen Dynamic Time Warping (DTW)-Distanzvarianten, angewendet. Zur weiteren Interpretation und Bewertung der Segmente werden ergänzende Merkmale hinzugezogen.

**Dateninhalt**

Grundlage der Untersuchung bildet das Financial Transactions Dataset von Kaggle, das umfassende Angaben zu finanziellen Transaktionen von Kartenkunden bereitstellt. Neben Informationen zu Transaktionszeitpunkt, Betrag, Händler und Art der Transaktion umfasst der Datensatz auch kartenbezogene Merkmale wie Kartentyp und Kreditlimit. Ein ergänzender MCC-Datensatz ordnet die Händler über standardisierte Branchencodes ein. Zu den rund 2.000 Kunden sind darüber hinaus demografische sowie kontobezogene Angaben verfügbar. Der Fraud-Datensatz, welcher den einzelnen Transaktionen ein Betrugskennzeichen zuweist, findet im Rahmen dieser Clusteranalyse keine Anwendung. <br>
Eine genaue Auflistung der in den Daten enthaltenen Variablen ist der Datei **[DATA_INFORMATION](../DATA_INFORMATION.md)** zu entnehmen.

Quelle: Vazquez, V. V. Financial Transactions Dataset: Analytics. Abgerufen am 20.02.2026 von https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets

***
## Phase 2: Datenverständnis
***

In [1]:
# Imports
import kagglehub
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import math
from pathlib import Path

# Imports Parameter
from parameter import INPUT_PATH, INPUT_ORIGIN_PATH

In [2]:
# Einstellung, dass alle Spalten eines Datensatzes angezeigt werden
pd.set_option("display.max_columns", None) 

### **Daten Download**

In [ ]:
# Daten von Kaggle downloaden 
# Quelle: Quelle: Vazquez, V. V. Financial Transactions Dataset: Analytics. Abgerufen am 20.02.2026 von https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets
kagglehub.dataset_download(
    "computingvictor/transactions-fraud-datasets",
    output_dir=INPUT_ORIGIN_PATH,
    force_download=True
)

### Transaktionsdaten

In [ ]:
path_temp = INPUT_ORIGIN_PATH + "/transactions_data.csv"
df_transactions = pd.read_csv(path_temp)  # Transaktionsdaten als DataFrame speichern
print(df_transactions.shape)  # Anzahl der Zeilen/Spalten ausgeben
df_transactions.head()  # Ersten 5 Einträge anzeigen

In [5]:
# ID-Spalte anpassen für eine eindeutige Zuordnung
df_transactions = df_transactions.rename(columns={"id": "id_transaction"})

### MCC-Daten

In [ ]:
path_temp = INPUT_ORIGIN_PATH + "/mcc_codes.json" 
# MCC-Daten als DataFrame speichern
df_mcc = pd.read_json(  
    path_temp, typ="series", encoding="utf-8"
)  
print(df_mcc.shape)  # Anzahl der Zeilen/Spalten ausgeben
df_mcc.head()  # Ersten 5 Einträge anzeigen

In [7]:
# Spaltenbezeichnung anpassen
df_mcc = df_mcc.rename_axis("mcc").reset_index(name="description") 

# MCC-Daten mit den Transaktionsdaten zusammenführen
df_transactions = df_transactions.merge(
    df_mcc, how="left", on="mcc"
)

### Kartendaten

In [ ]:
path_temp = INPUT_ORIGIN_PATH + "/cards_data.csv"
df_cards = pd.read_csv(path_temp)  # Kartendaten als DataFrame speichern
print(df_cards.shape)  # Anzahl der Zeilen/Spalten ausgeben
df_cards.head()  # Ersten 5 Einträge anzeigen

In [9]:
# ID-Spalte anpassen für eine eindeutige Zuordnung
df_cards = df_cards.rename(columns={"id": "id_cards"})

# Kartendaten mit den Transaktionsdaten zusammenführen
df_transactions = df_transactions.merge(
    df_cards, how="left", on="client_id"
)

### Kundendaten

In [ ]:
path_temp = INPUT_ORIGIN_PATH + "/users_data.csv"
df_user = pd.read_csv(path_temp)  # Kundeninformationen als DataFrame speichern
print(df_user.shape)  # Anzahl der Zeilen/Spalten ausgeben
df_user.head()  # Ersten 5 Einträge anzeigen

In [11]:
# ID-Spalte anpassen für eine eindeutige Zuordnung
df_user = df_user.rename(columns={"id": "id_user"})

# Kundendaten mit den Transaktionsdaten zusammenführen
df_transactions = df_transactions.merge(
    df_user, how="left", left_on="client_id", right_on="id_user"
) 

### **Daten Speicherung**

In [18]:
INPUT_PATH = Path(INPUT_PATH)
path_temp = INPUT_PATH / "data_acquisition.csv"
INPUT_PATH.mkdir(parents=True, exist_ok=True) # Ordner erstellen (falls nicht vorhanden)
df_transactions.to_csv(path_temp, index=False)

***
## Phase 2: Explorative Datenanalyse (EDA)
***

### Kategorische Variablen

In [ ]:
# Ausgabe der kategorischen Variablennamen (10 Variablen pro Zeile)
categorical_var = [
    col for col in df_transactions 
    if df_transactions[col].dtype == "object"
]
print("Kategorische Variablen:")
for var in range(0, len(categorical_var), 10):
    print(", ".join(categorical_var[var:var+10]))

In [ ]:
# Anzahl der unterschiedlichen Werte pro kategorischer Variable ausgeben
for var in categorical_var:
    n_unique_values = df_transactions[var].nunique()
    if n_unique_values < 100:
        print(df_transactions[var].value_counts())
        print("")
    # Für Variablen die mehr als 100 Ausprägungen, wird nur die Anzahl der Ausprägungen ausgegeben 
    else: 
        print(var)
        print(f"Anzahl der verschiedenen Werte: {n_unique_values}")
        print("")

In [ ]:
# Zeitraum anzeigen
print(f"Minimales Datum {df_transactions["date"].min()}")
print(f"Maximale Datum {df_transactions["date"].max()}")

In [ ]:
# Bar Plot der kategorischen Variablen (der ersten 50 Einträge)

categorical_var = [
    col for col in categorical_var 
    if col not in ["date"]  # Date nicht verwenden, da zu viele unterschiedliche Werte vorliegen
] 

# Anzeige: 3 Plots nebeneinander
n_cols = 3  
n_rows = math.ceil(len(categorical_var) / n_cols)
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(15, 5*n_rows)
)
axes = axes.flatten()  
counter = 0

for var in categorical_var:
    # Anzeige auf die ersten 50 Einträge einschränken
    count_temp = df_transactions[var].value_counts().head(50) 
    ax = axes[counter]
    count_temp.plot(kind="bar", ax=ax)
    ax.set_title(var)
    ax.set_xlabel("")
    ax.set_ylabel("Anzahl")
    ax.tick_params(axis="x", rotation=45)

    counter += 1

# Leere Subplots entfernen
for j in range(counter, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

**Interpretation**

Der Datensatz beinhaltet Transaktionsdaten über einen Zeitraum von neun Jahren, beginnend am 01.01.2010 (**date**).

Die meisten Transaktionsbeträge (**amount**) konzentrieren sich im niedrigen Bereich, wobei Beträge um 100 Dollar am häufigsten vorkommen. Zudem tritt der Wert 0 ebenfalls relativ häufig auf. Die Variable wird mit einem Dollarzeichen dargestellt. Dieses sollte entfernt und die Variable in eine numerische umgewandelt werden.

Die Grafik zu der Variable **use_chip** Swipe Transaction, Chip Transaction und Online Transaction. Da diese Variable lediglich die technische Abwicklung der Transaktion beschreibt und keinen Informationsgehalt für das Clustering besitzt, sollte sie entfernt werden.

Die Variable **merchant_city** weist viele verschiedene Ausprägungen auf, von denen keine besonders häufig vorkommt. Lediglich der Wert „online" sticht am linken Rand hervor. Da die Stadt aus der Postleitzahl (zip) abgeleitet werden kann und die hohe Kardinalität keine sinnvolle Zeitreihenbildung erlaubt, wird diese Variable nicht weiter betrachtet.

Auch die Grafik zur Variable **merchant_state** zeigt viele verschiedene Werte in einem abfallenden Balkendiagramm. Am häufigsten tritt der Wert CA (California) auf, gefolgt von weiteren US-Bundesstaaten mit deutlich geringerer Häufigkeit. Für die Weiterverwendung sollte diese Variable in übergeordnete Kategorien zusammengefasst werden.

Das Balkendiagramm der Variable **errors** zeigt, dass die überwiegende Mehrheit der Transaktionen keinen Fehler aufweist. Unter den fehlerhaften Transaktionen tritt der Wert „Insufficient Balance" am häufigsten auf.

Die Variable **description** enthält viele unterschiedliche Händlerbeschreibungen. Einige wenige Kategorien wie „Online Shopping" oder „Subscription Services" kommen häufiger vor, während die meisten Beschreibungen nur selten auftreten. Für die Weiterverarbeitung sollten hier übergeordnete Kategorien gebildet werden.

In den Daten gibt es vier Kartenhersteller (**card_brand**). Die meisten Transaktionen wurden über Mastercard und Visa abgewickelt, während Amex und Discover deutlich seltener vorkommen.

Aus der Variable **card_type** geht hervor, dass die meisten Karten Debitkarten sind. Da es sich um eine statische Karteneigenschaft handelt, sollte die Variable entfernt werden.

Die Variablen **expires** und **acct_open_date** besitzen individuelle Werte. Beide sind feste Karteneigenschaften und sollten für die Clusteranalyse entfernt werden.

Die Mehrheit der Karten besitzt einen Chip (**has_chip**). Da es sich um eine statische Karteneigenschaft handelt, sollte diese Variable entfernt werden.

Die Variable **credit_limit** wird mit einem Dollarzeichen dargestellt. Dieses sollte entfernt werden und in eine numerische Variable geändert werden. Die häufigsten Werte liegen im Bereich um 10.600 Dollar, gefolgt vom Wert 0 (kein Limit)

Es gibt ausschließlich Karten, die nicht im Darkweb auftauchen (**card_on_dark_web**). Da die Variable nur eine einzige Ausprägung besitzt, hat sie keinen Informationsgehalt und sollte entfernt werden.

Bei der Geschlechterverteilung (**gender**) gibt es keine Auffälligkeiten. Die beiden Geschlechter sind nahezu gleichverteilt.

Die Variable **address** besitzt individuelle Werte. Da die Adresse ein statisches, kundenspezifisches Merkmal ist und für die geografische Zuordnung stattdessen die Variable zip verwendet wird, sollte diese Variable entfernt werden.

Die Variable **per_capita_income** zeigt eine breite, relativ flache Verteilung ohne starke Ausreißer. Die Variable wird mit einem Dollarzeichen dargestellt. Dieses sollte entfernt und die Variable in eine numerische umgewandelt werden.

Auch die Variable **yearly_income** zeigt eine breite, flache Verteilung ohne starke Ausreißer. Die Variable wird mit einem Dollarzeichen dargestellt. Dieses sollte entfernt und die Variable in eine numerische umgewandelt werden.

Bei den meisten Kunden liegt keine oder nur eine geringe Verschuldung (**total_debt**) vor. Der Großteil der Werte konzentriert sich nahe Null, dennoch gibt es vereinzelt Kunden mit hoher Verschuldung. Die Variable wird mit einem Dollarzeichen dargestellt und sollte in eine numerische Variable umgewandelt werden.

### Numerische Variablen

In [ ]:
# Ausgabe der numerischen Variablennamen (ohne ID-Spalten) 
numerical_var = [
    col for col in df_transactions 
    if df_transactions[col].dtype != "object"
]

print("Numerischen Variablen:")
for var in range(0, len(numerical_var), 10):
    print(", ".join(numerical_var[var:var+10]))

In [ ]:
# Statistische Kennzahlen der numerischen Variablen ausgeben
df_transactions[numerical_var].describe().T

In [ ]:
# Histogramm der numerischen Variablen

# ID-Variablen nicht anzeigen, da zu viele unterschiedliche Werte und ID-Variablen beinhalten keinen relevante Information
id_cols = [
    "id_transaction", "client_id", "card_id",
    "merchant_id", "id_cards", "id_user", "card_number",
]
numerical_var = [
    col for col in numerical_var 
    if col not in id_cols  # ID-Spalten nicht betrachten (kein Informationsgehalt)
]  

n_cols = 5
n_rows = math.ceil(len(numerical_var) / n_cols)
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(20, 4*n_rows)
)
axes = axes.flatten()

for i, col in enumerate(numerical_var):
    axes[i].hist(df_transactions[col], bins=100)
    axes[i].set_title(col)
    axes[i].set_ylabel("Häufigkeit")
    axes[i].xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
    axes[i].tick_params(axis="x", rotation=45)

# Leere Subplots entfernen
for j in range(len(numerical_var), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot der numerischen Variablen

n_cols = 5 # Anzahl der Plots
n_rows = math.ceil(len(numerical_var) / n_cols)
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(20, 4*n_rows)
)
axes = axes.flatten()

for i, col in enumerate(numerical_var):
    axes[i].boxplot(df_transactions[col].dropna()) # Boxplot
    axes[i].set_title(col)
    axes[i].yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
    
# Leere Subplots entfernen
for j in range(len(numerical_var), len(axes)):
    axes[j].axis("off")
    
plt.tight_layout()
plt.show()

**Interpretation**

Die Variable **zip** sollte als String gespeichert werden, da es sich um einen kategorialen Identifier handelt. Die Verteilung ist breit gestreut und zeigt keine starke regionale Häufung. 

Der Boxplot zur Variable **mcc** zeigt eine Konzentration der Werte um 5.000–6.000 mit zahlreichen Ausreißern nach oben und unten. Das Histogramm verdeutlicht, dass ein einzelner MCC-Code stark dominiert, während die übrigen Codes deutlich seltener auftreten. Da der MCC-Code die numerische Entsprechung der Beschreibungs-Variable (description) ist, wird für die Analyse lediglich die Beschreibungs-Variable beibehalten.

Der Boxplot zur Variable **cvv** zeigt eine nahezu gleichmäßige Verteilung über den gesamten Wertebereich, welche durch das Histogramm bestätigt wird. Der CVV ist eine zufällig generierte Sicherheitsnummer ohne Bezug zum Transaktionsverhalten und liefert keinen Beitrag zur Erkennung zeitlicher Verhaltensmuster. Diese Variable sollte entfernt werden.

Aus den Grafiken zu der Variable **num_cards_issued** geht hervor, dass die meisten Kunden ein bis zwei Karten pro Konto besitzen. 

Die häufigsten letzten Änderungen des Kartenpins (**year_pin_last_changed**) fanden zwischen 2010 und 2015 statt.

Die Grafiken zur **current_age** zeigen, dass die Kunden überwiegend mittleren bis höheren Alters sind (40–65 Jahre), während jüngere Kunden unter 30 und sehr alte über 80 eher unterrepräsentiert sind. Da das Alter aus dem Geburtsjahr abgeleitet werden kann, sind beide Variablen redundant. Es wird lediglich die Variable birth_year beibehalten und current_age entfernt.

Durch die Variable **retirement_age** geht hervor, dass die meisten Kunden zwischen dem Alter von 65 und 67 in die Rente gehen. Zudem weist die Variable eine geringe Varianz auf. Da es sich um ein weitgehend standardisiertes Rentenalter handelt, liefert diese Variable keinen Beitrag zur Erkennung unterschiedlicher Verhaltensmuster und sollte entfernt werden.

Die Geburtsjahre (**birth_year**) häufen sich in den Jahren 1950 bis 1975, was konsistent mit dem beobachteten Altersprofil ist. 

Es liegt eine gleichmäßige Verteilung über alle 12 Monate der Variable **birth_month** vor. Lediglich in den Sommermonaten sinkt die Anzahl etwas.

Die geografischen Variablen **latitude** und **longitude** sind statische Kundenmerkmale und liefern keinen Beitrag zur Analyse zeitlicher Verhaltensmuster. Für die geografische Zuordnung wird stattdessen die Variable zip verwendet. Beide Variablen sollten entfernt werden.

Die Variable **credit_score** häuft sich bei Werten von 650 bis 750, was auf Standardkunden mit guter Bonität hindeutet. Gleichzeitig ist ein sichtbarer Ausreißerblock bei etwa 500 erkennbar, der eine Minderheit von Kunden mit schlechter Bonität repräsentiert.

Die Grafiken zu der Variable **num_credit_cards** zeigen, dass der Median bei etwa 4 Kreditkarten liegt, wobei die häufigsten Werte zwischen 1 und 5 auftreten. Vereinzelte Ausreißer sind bei 9 Kreditkarten erkennbar.

### Korrelationsmatrix

In [ ]:
# Korrelationsmatrix der numerischen Variablen 
correlation_matrix = (
    df_transactions[numerical_var]
    .corr(method="pearson")
    .round(2) )

# Plot: Korrelationsmatrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,  # Werte anzeigen        
    fmt=".2f",  # Auf 2 Nachkommastellen anpassen
    ax=ax
)
ax.set_title("Korrelationsmatrix (numerischen Werte)")
plt.tight_layout()
plt.show()

**Interpretation**

Die Variablen **current_age** und **birth_year** weisen eine sehr hohe negative Korrelation (-1) auf. Dies ist erwartbar, da das Alter direkt aus dem Geburtsjahr berechnet wird. Beide Variablen enthalten somit dieselbe Information. Die Variable **current_age** sollte gelöscht werden, da man aus der Variable **birth_year** auch das Alter bestimmen kann.

Die Variablen **longitude** und **zip** weisen ebenfalls eine relativ starke negative Korrelation (-0.86) auf. Dies ist ebenfalls plausibel, da Postleitzahlen geografisch organisiert sind und daher mit dem Längengrad zusammenhängen. Regionen mit ähnlichen Postleitzahlen liegen häufig auch in ähnlichen geografischen Bereichen.

Eine moderate positive Korrelation (0.41) weisen die Variablen **num_credit_cards** und **current_age** auf. Dies lässt sich so interpretieren, dass ältere Personen im Durchschnitt mehr Kreditkarten besitzen. Eine mögliche Erklärung ist eine längere Kredit- und Bankhistorie, die im Laufe des Lebens zur Nutzung mehrerer Finanzprodukte führt.

Entsprechend zeigt sich auch zwischen **num_credit_cards** und **birth_year** eine moderate negative Korrelation (-0.41). Dies ist konsistent mit der vorherigen Beobachtung, da ein höheres Geburtsjahr jüngere Personen beschreibt, die durchschnittlich weniger Kreditkarten besitzen.

Zwischen **credit_score** und **num_credit_cards** besteht eine schwache positive Korrelation (0.22). Dies könnte darauf hindeuten, dass Personen mit mehreren Kreditkarten tendenziell einen etwas höheren Kredit-Score aufweisen, möglicherweise aufgrund einer längeren Kreditgeschichte oder einer stärkeren Nutzung von Kreditprodukten.

Die meisten übrigen Variablenpaare weisen hingegen nur sehr geringe Korrelationen auf (Werte nahe 0), was darauf hindeutet, dass zwischen diesen Variablen kein starker linearer Zusammenhang besteht.

### Fehlende Werte (Grafik)

In [ ]:
# Missing Werte
msno.matrix(df_transactions, figsize=(10,8))
plt.show()

**Interpretation**

Aus der Grafik geht hervor, dass die Variable **errors** in fast allen Datensätzen fehlende Werte aufweist. Die fehlenden Einträge deuten darauf hin, dass in den entsprechenden Transaktionen kein Fehler aufgetreten ist. Lediglich in fünf Fällen liegt ein tatsächlicher Wert vor, was darauf hinweist, dass dort ein Fehler bei der Transaktion aufgetreten ist.

Darüber hinaus zeigen die Variablen **zip** und **merchant_state** fehlende Werte auf, die weitgehend bei denselben Datensätzen auftreten. Insgesamt handelt es sich hierbei jedoch nur um vereinzelte fehlende Einträge, sodass die Anzahl der betroffenen Fälle vergleichsweise gering ist und keine systematische Häufung erkennbar ist.


***

In [19]:
# # Alle Variablen und Objekte aus dem Arbeitsspeicher löschen
%reset -f

***
***